In [ ]:
import os
import json
import requests
import pandas as pd
from pathlib import Path
from typing import Dict, Any, List, Optional, Tuple, Set

from opensearchpy import OpenSearch
from product_search.ingestion import (
    create_bm25_index, bulk_ingest_bm25,
    create_hnsw_index, bulk_ingest_hnsw,
)

In [2]:
project_dir   = Path(os.getcwd()).parent
data_dir      = project_dir / "Data" / "RAW"
processed_dir = project_dir / "Data" / "PROCESSED"

In [ ]:
bm25_index_name = "bm25_index"
hnsw_index_name = "hnsw_index"

batch_size = 1000
vector_field = "embedding"

# Load dev.env
_env_file = Path(os.getcwd()).parent / "dev.env"
if _env_file.exists():
    for _line in _env_file.read_text().splitlines():
        _line = _line.strip()
        if _line and not _line.startswith("#") and "=" in _line:
            _key, _, _val = _line.partition("=")
            os.environ.setdefault(_key.strip(), _val.strip().strip('"'))
    print("[INFO] dev.env loaded")

CLOUDRUN_URL = os.getenv("CLOUDRUN_URL", "http://localhost:8001")
print(f"[INFO] Embedding service URL: {CLOUDRUN_URL}")

In [4]:
client = OpenSearch(
    hosts=[{"host": "localhost", "port": 9200}],
    use_ssl=False,
    verify_certs=False,
    ssl_show_warn=False,
    timeout=60,
)

In [5]:
# load datasets
with open(processed_dir / "product_store.json", "r", encoding="utf-8") as f:
    product_store = json.load(f)

with open(processed_dir / "train_qrels.json", "r", encoding="utf-8") as f:
    train_qrels_dict = json.load(f)

with open(processed_dir / "test_qrels.json", "r", encoding="utf-8") as f:
    test_qrels_dict = json.load(f)

In [ ]:
# Probe the embedding service to discover the embedding dimension
resp = requests.post(
    f"{CLOUDRUN_URL}/encode",
    json={"texts": ["probe"], "normalize": True},
    timeout=30,
)
resp.raise_for_status()
dim = len(resp.json()["embeddings"][0])
print(f"[INFO] Embedding service: {CLOUDRUN_URL}")
print(f"[INFO] Embedding dim    : {dim}")

In [7]:
product_ids = sorted(list(product_store.keys()))
print(f'Number of products in store: {len(product_ids)}')

Number of products in store: 23052


In [8]:
if client.indices.exists(index=bm25_index_name):
    client.indices.delete(index=bm25_index_name)

if client.indices.exists(index=hnsw_index_name):
    client.indices.delete(index=hnsw_index_name)

In [9]:
create_bm25_index(client, bm25_index_name)
print(f"[OK] BM25 index ready: {bm25_index_name}")

[OK] BM25 index ready: bm25_index


In [10]:
bulk_ingest_bm25(
    client,
    index_name=bm25_index_name,
    product_store=product_store,
    product_ids=product_ids,
    batch_size=batch_size,
)
print(f"[OK] BM25 ingested: {bm25_index_name} (docs={client.count(index=bm25_index_name)['count']})")

BM25 ingest → bm25_index:   0%|          | 0/23052 [00:00<?, ?docs/s]

BM25 ingest → bm25_index: 100%|██████████| 23052/23052 [00:03<00:00, 6733.21docs/s]


[OK] BM25 ingested: bm25_index (docs=23052)


In [11]:
create_hnsw_index(
    client,
    hnsw_index_name,
    dim=dim,
    vector_field=vector_field,
    space_type="cosinesimil",
    engine="faiss",
)
print(f"[OK] HNSW index ready: {hnsw_index_name} (dim={dim})")

[OK] HNSW index ready: hnsw_index (dim=384)


In [ ]:
bulk_ingest_hnsw(
    client,
    index_name=hnsw_index_name,
    product_store=product_store,
    product_ids=product_ids,
    embedding_service_url=CLOUDRUN_URL,
    dim=dim,
    vector_field=vector_field,
    batch_size=batch_size,
    encode_batch_size=batch_size,
    normalize_embeddings=True,
)
print(f"[OK] HNSW ingested: {hnsw_index_name} (docs={client.count(index=hnsw_index_name)['count']})")